# LLM Baselines on SG Synthetic Dataset (Zero-Shot & Few-Shot)

Evaluate multiple LLMs on the SG synthetic hate speech test set **before any fine-tuning**.

**Task:** 3-class classification — `hate`, `offensive`, `normal`  
**Dataset:** `data/singlish/synthetic/20_terms/split_60_20_20/test.jsonl` (108 rows)  
**Modes:** Zero-shot and few-shot (2 examples per class in the system prompt)  

### Models tested
| Model | Via | Cost |
|-------|-----|------|
| `meta-llama/llama-guard-3-8b` | OpenRouter | ~$0.20/1M |
| `anthropic/claude-3-5-haiku` | OpenRouter | ~$0.80/1M |
| `google/gemini-2.0-flash-001` | OpenRouter | ~$0.10/1M |
| `deepseek/deepseek-chat` | OpenRouter | ~$0.14/1M |

### Setup
```bash
uv venv .venv-openrouter && source .venv-openrouter/bin/activate
uv add openrouter scikit-learn pandas python-dotenv tqdm ipykernel
python -m ipykernel install --user --name=openrouter
```
Add `OPENROUTER_API_KEY=...` to `.env`.

In [ ]:
import os
import json
import time
import csv
import pandas as pd
from tqdm import tqdm
from openrouter import OpenRouter
from sklearn.metrics import classification_report, f1_score
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

# ── Models to test (uncomment to run) ──────────────────────────────────
MODELS = [
    'meta-llama/llama-guard-3-8b',
    'anthropic/claude-3-5-haiku',
    'google/gemini-2.0-flash-001',
    'deepseek/deepseek-chat',
]

CHECKPOINT_INTERVAL = 10
RESULTS_FILE = 'results/model_scores_sg.csv'

if not os.getenv('OPENROUTER_API_KEY'):
    raise ValueError('OPENROUTER_API_KEY not found in .env')

client = OpenRouter(api_key=os.getenv('OPENROUTER_API_KEY'))

# Quick connection check
resp = client.chat.send(
    model=MODELS[1],
    messages=[{'role': 'user', 'content': "Reply 'ok'"}],
    max_tokens=5,
)
print(f'API OK: {resp.choices[0].message.content.strip()}')

## 1. Load SG Test Data

In [ ]:
from collections import Counter

LABEL_NAMES = ['hate', 'offensive', 'normal']

TEST_PATH = 'data/singlish/synthetic/20_terms/split_60_20_20/test.jsonl'
TRAIN_PATH = 'data/singlish/synthetic/20_terms/split_60_20_20/train.jsonl'

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

test_data = load_jsonl(TEST_PATH)
train_data = load_jsonl(TRAIN_PATH)

print(f'Test set: {len(test_data)} rows')
print(f'Label distribution: {dict(Counter(d["label"] for d in test_data))}')
print(f'\nSample row:')
print(json.dumps(test_data[0], indent=2))

## 2. Prompt Templates

In [ ]:
# ── Zero-shot prompt ──────────────────────────────────────────────────
ZERO_SHOT_SYSTEM = """You are a Singapore content moderation expert. Classify the following text into exactly one of these categories:

- 'hate': targets a person or group based on race, religion, nationality, or identity. Includes racial slurs (e.g. 'ah tiong', 'bangla'), xenophobic statements, or derogatory generalisations about an ethnic group.
- 'offensive': rude or inappropriate language (e.g. Singlish expletives like 'nabeh', 'cb', 'wah lan') but NOT directed at a specific identity group.
- 'normal': neutral, benign, or everyday content — including casual Singlish like 'lah', 'leh', 'shiok'.

Reply with ONLY the label in lowercase: hate, offensive, or normal. No explanation."""

# ── Few-shot examples (2 per class, drawn from training set) ──────────
FEW_SHOT_EXAMPLES = [
    # hate
    {'text': 'those ah tiong always come here take our jobs lor', 'label': 'hate'},
    {'text': 'all bangla workers dirty and smelly one, should ban them', 'label': 'hate'},
    # offensive
    {'text': 'nabeh you damn stupid lah, cannot do simple thing', 'label': 'offensive'},
    {'text': 'wah lan eh, you spoil my mood again', 'label': 'offensive'},
    # normal
    {'text': 'wah lao the mee pok today damn shiok sia', 'label': 'normal'},
    {'text': 'eh later we go makan together can?', 'label': 'normal'},
]

few_shot_block = '\n'.join(
    f'Text: "{ex["text"]}"\nLabel: {ex["label"]}'
    for ex in FEW_SHOT_EXAMPLES
)

FEW_SHOT_SYSTEM = ZERO_SHOT_SYSTEM + f'\n\nExamples:\n{few_shot_block}'

print('Zero-shot system prompt:')
print(ZERO_SHOT_SYSTEM)
print('\n' + '='*60)
print('\nFew-shot adds these examples:')
print(few_shot_block)

## 3. Classification Function

In [ ]:
def normalize_label(raw):
    """Parse model output to one of hate/offensive/normal."""
    pred = str(raw).lower().strip()
    if pred.startswith('hate'):
        return 'hate'
    if pred.startswith('off'):
        return 'offensive'
    if pred.startswith('norm'):
        return 'normal'
    return 'invalid'


def classify_sg(text, model_name, mode='zero_shot', retries=3):
    """
    Classify text using the specified model and mode.
    mode: 'zero_shot' | 'few_shot'
    Returns: 'hate' | 'offensive' | 'normal' | 'invalid' | 'api_error'
    """
    system = ZERO_SHOT_SYSTEM if mode == 'zero_shot' else FEW_SHOT_SYSTEM

    for attempt in range(retries):
        try:
            resp = client.chat.send(
                model=model_name,
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': f'Text: "{text}"'},
                ],
                temperature=0,
                max_tokens=10,
            )
            raw = resp.choices[0].message.content.strip()
            return normalize_label(raw)
        except Exception as e:
            if attempt == retries - 1:
                print(f'\nAPI error after {retries} attempts: {e}')
                return 'api_error'
            time.sleep(2 ** attempt)  # Exponential backoff


# Smoke test
sample = test_data[0]['text']
print(f'Sample text: "{sample[:60]}"')
print(f'Zero-shot ({MODELS[1]}): {classify_sg(sample, MODELS[1], "zero_shot")}')
print(f'Few-shot  ({MODELS[1]}): {classify_sg(sample, MODELS[1], "few_shot")}')

## 4. Results Logging Helper

In [ ]:
os.makedirs('results', exist_ok=True)

def save_result(model_name, mode, y_true, y_pred):
    """Compute metrics and append a row to the results CSV."""
    valid_mask = [p not in ('api_error', 'invalid') for p in y_pred]
    yt = [y for y, v in zip(y_true, valid_mask) if v]
    yp = [p for p, v in zip(y_pred, valid_mask) if v]

    n_errors = sum(1 for v in valid_mask if not v)
    report = classification_report(yt, yp, labels=LABEL_NAMES, output_dict=True, zero_division=0)

    row = {
        'timestamp': datetime.now().isoformat(),
        'model_name': model_name,
        'model_type': f'api ({mode})',
        'task': f'sg_3class_{mode}',
        'test_samples': len(yt),
        'api_errors': n_errors,
        'accuracy': round(report['accuracy'], 4),
        'macro_f1': round(report['macro avg']['f1-score'], 4),
        'macro_precision': round(report['macro avg']['precision'], 4),
        'macro_recall': round(report['macro avg']['recall'], 4),
        'hate_f1': round(report.get('hate', {}).get('f1-score', 0), 4),
        'offensive_f1': round(report.get('offensive', {}).get('f1-score', 0), 4),
        'normal_f1': round(report.get('normal', {}).get('f1-score', 0), 4),
    }

    file_exists = os.path.exists(RESULTS_FILE)
    with open(RESULTS_FILE, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

    print(f'  Saved to {RESULTS_FILE}')
    return report, yt, yp


print('Results logger ready.')

## 5. Run Inference

Runs all configured models in both zero-shot and few-shot modes. Adjust `MODELS` and `MODES` to run a subset.

In [ ]:
# Adjust these to run a subset
MODELS_TO_RUN = MODELS          # or e.g. ['anthropic/claude-3-5-haiku']
MODES_TO_RUN = ['zero_shot', 'few_shot']

all_results = {}  # (model, mode) -> (y_true, y_pred)

for model_name in MODELS_TO_RUN:
    for mode in MODES_TO_RUN:
        key = (model_name, mode)
        print(f'\n{"="*60}')
        print(f'Model: {model_name.split("/")[-1]}  |  Mode: {mode}')
        print('='*60)

        y_true, y_pred = [], []

        for i, row in enumerate(tqdm(test_data, desc=f'{mode}')):
            pred = classify_sg(row['text'], model_name, mode)
            y_true.append(row['label'])
            y_pred.append(pred)

            if (i + 1) % CHECKPOINT_INTERVAL == 0:
                valid_so_far = [p for p in y_pred if p not in ('api_error', 'invalid')]
                correct = sum(t == p for t, p in zip(y_true, y_pred) if p not in ('api_error', 'invalid'))
                acc = correct / len(valid_so_far) if valid_so_far else 0
                print(f'  [{i+1}/{len(test_data)}] running acc={acc:.2f}, errors={y_pred.count("api_error")}')

        all_results[key] = (y_true, y_pred)
        report, yt, yp = save_result(model_name, mode, y_true, y_pred)
        print(f'  Macro F1: {report["macro avg"]["f1-score"]:.4f}  |  Accuracy: {report["accuracy"]:.4f}')

print('\nAll inference runs complete.')

## 6. Results & Comparison

In [ ]:
from sklearn.metrics import confusion_matrix

for (model_name, mode), (y_true, y_pred) in all_results.items():
    valid_mask = [p not in ('api_error', 'invalid') for p in y_pred]
    yt = [y for y, v in zip(y_true, valid_mask) if v]
    yp = [p for p, v in zip(y_pred, valid_mask) if v]
    n_err = sum(1 for v in valid_mask if not v)

    print(f'\n{"="*65}')
    print(f'  {model_name.split("/")[-1]}  [{mode}]  ({len(yt)} valid / {n_err} errors)')
    print('='*65)
    print(classification_report(yt, yp, labels=LABEL_NAMES, zero_division=0))

print('\n' + '='*65)
print('LEADERBOARD — Macro F1 on SG Test Set')
print('='*65)

rows = []
for (model_name, mode), (y_true, y_pred) in all_results.items():
    yt = [y for y, p in zip(y_true, y_pred) if p not in ('api_error', 'invalid')]
    yp = [p for p in y_pred if p not in ('api_error', 'invalid')]
    if yt:
        f1 = f1_score(yt, yp, labels=LABEL_NAMES, average='macro', zero_division=0)
        rows.append({'Model': model_name.split('/')[-1], 'Mode': mode, 'Macro F1': round(f1, 4)})

lb = pd.DataFrame(rows).sort_values('Macro F1', ascending=False).reset_index(drop=True)
lb.index += 1
print(lb.to_string())

## 7. Error Analysis — Where Models Fail on Singlish

In [ ]:
# Pick the best-performing model+mode to analyse
best_key = lb.iloc[0][['Model', 'Mode']].values
best_model = next(k for k in all_results if k[0].split('/')[-1] == best_key[0] and k[1] == best_key[1])

y_true, y_pred = all_results[best_model]

print(f'Error analysis: {best_model[0].split("/")[-1]} [{best_model[1]}]')
print(f'{"True":<11} {"Pred":<11} {"Term":<18} Text')
print('-'*100)

errors_by_type = Counter()
for d, t, p in zip(test_data, y_true, y_pred):
    if t != p and p not in ('api_error', 'invalid'):
        errors_by_type[f'{t}->{p}'] += 1
        print(f'{t:<11} {p:<11} {d.get("source_term", "?"):<18} {d["text"][:70]}')

print(f'\nConfusion types: {dict(errors_by_type)}')